# Prepare EGMS PS density for training

This notebook automates two steps of the EGMS workflow. The download and QGIS density conversion stay manual.

1. **Manual.** Draw the AOI on the EGMS website and save the archive link list.
2. **This notebook.** Filter that list by orbit.
3. **Manual.** Use the filtered file as input to the EGMS download stream, then unzip shapefiles.
4. **This notebook.** Merge split `*_partN.shp` files into `{id}_merged.shp`.
5. **Manual (QGIS).** Convert merged PS points to a density raster and save it under `train.input.mask_egms_dir`.

Set `prepare.egms_links_txt` and `prepare.egms_shp_dir` in `config/paths.yaml` before running.

In [1]:
from pathlib import Path

import geopandas as gpd

from ps_density.paths import load_paths

_cfg = load_paths()
links_txt = Path(_cfg.prepare.egms_links_txt)
shp_dir = Path(_cfg.prepare.egms_shp_dir)
mask_dir = Path(_cfg.train.input.mask_egms_dir)
print("links:", links_txt)
print("shapefiles:", shp_dir)
print("density output:", mask_dir)

links: /mnt/e/global_coh_map/00_data_to_be_published/input/egms/egms-archive-links.txt
shapefiles: /mnt/e/global_coh_map/00_data_to_be_published/input/egms/shp
density output: /mnt/e/global_coh_map/00_data_to_be_published/input/mask/mask_EGMS


## 1. Manual: download the EGMS link list

On the EGMS portal, mark the area of interest and download the archive links list (a `.txt` file of `https://egms.land.copernicus.eu/...` URLs). The study used calibrated data (level 2b).

The associated dataset published on zenodo contains an example of such a file so that the notebook can be tested. 

Point `prepare.egms_links_txt` in config/paths.yaml at that file. Do not run the next cell until the file exists.

## 2. Filter the link list by orbit

Keeps lines whose product name contains one of the orbit prefixes. The Netherlands descending set used for the paper is listed first; the Rome example is commented.

Writes a **new** `*_filtered.txt` next to the original. The original download is not overwritten.

In [3]:
# Netherlands descending (paper training mask)
orbits = ["EGMS_L2b_037", "EGMS_L2b_110", "EGMS_L2b_008", "EGMS_L2b_139"]
# Rome descending example:
# orbits = ["EGMS_L2b_095", "EGMS_L2b_022"]

with links_txt.open(encoding="utf-8") as handle:
    lines = handle.readlines()

filtered = [line for line in lines if any(orbit in line for orbit in orbits)]
filtered.sort()

filtered_txt = links_txt.with_name(f"{links_txt.stem}_filtered.txt")
filtered_txt.write_text("".join(filtered), encoding="utf-8")
print(f"Kept {len(filtered)} of {len(lines)} links")
print("Wrote", filtered_txt)

Kept 31 of 64 links
Wrote /mnt/e/global_coh_map/00_data_to_be_published/input/egms/egms-archive-links_filtered.txt


## 3. Manual: download shapefiles with the EGMS stream

Use `*_filtered.txt` as the input list for the EGMS stream (https://egmstream.unifi.it/, https://doi.org/10.3390/rs15102581) to download the shapefiles. Unzip the L2b products so that `.shp` files sit in `prepare.egms_shp_dir` (`shp_dir` above).

This notebook does not download or unzip those archives.

## 4. Merge split shapefile parts

Groups `*_partN.shp` by the 24-character product id and writes `{id}_merged.shp`. Products that already have a merged file are skipped.

In [4]:
shp_files = sorted(path.name for path in shp_dir.glob("*.shp"))
product_ids = sorted({name[:24] for name in shp_files})

for product_id in product_ids:
    parts = [name for name in shp_files if name.startswith(product_id)]
    print("Currently merged:", product_id)

    if not any("part" in name for name in parts):
        print("No parts for this file, moving to the next one.")
        continue
    if any("merged" in name for name in parts):
        print("This file is already merged, moving to the next one.")
        continue

    frames = [gpd.read_file(shp_dir / name) for name in parts]
    merged = gpd.pd.concat(frames)
    merged.to_file(shp_dir / f"{product_id}_merged.shp")

Currently merged: EGMS_L2b_037_0755_IW2_VV
No parts for this file, moving to the next one.
Currently merged: EGMS_L2b_037_0756_IW2_VV
No parts for this file, moving to the next one.
Currently merged: EGMS_L2b_037_0756_IW3_VV
No parts for this file, moving to the next one.
Currently merged: EGMS_L2b_037_0757_IW2_VV
No parts for this file, moving to the next one.
Currently merged: EGMS_L2b_037_0757_IW3_VV
No parts for this file, moving to the next one.
Currently merged: EGMS_L2b_037_0758_IW2_VV
No parts for this file, moving to the next one.
Currently merged: EGMS_L2b_037_0758_IW3_VV
No parts for this file, moving to the next one.
Currently merged: EGMS_L2b_037_0759_IW2_VV
No parts for this file, moving to the next one.
Currently merged: EGMS_L2b_037_0759_IW3_VV
No parts for this file, moving to the next one.
Currently merged: EGMS_L2b_037_0760_IW2_VV
No parts for this file, moving to the next one.
Currently merged: EGMS_L2b_037_0760_IW3_VV
No parts for this file, moving to the next one.

## 5. Manual (QGIS): PS points to density raster

This can be done in QGIS or other geospatial software. Save the GeoTIFF into path defined as `train.input.mask_egms_dir` in paths.yaml.

### Grid preparation

1. **Vector / Research Tools / Extract Layer Extent** as a batch process for each merged layer in the region.
2. **Vector / Data Management Tools / Merge Vector Layers** — merge those extents into one.
3. **Vector / Geoprocessing Tools / Dissolve** — dissolve polygons into one.
4. **Vector / Research Tools / Create Grid** using the dissolved extent. Spacing is 300 arcsec (`0.0833333`).

### Density map

In QGIS, **Processing → Graphical Modeler**, rebuild the graph below. It counts PS points into a 3-arcsec polygon grid. Rasterize **after** the model; do not rasterize inside the modeler.

Run it once per merged PS shapefile. The extent-polygon-grid input is the 300-arcsec grid from the previous step (for example `grid_Italy_300arcsec.shp`).

**Model inputs**

- **Input vector layer with PS** — vector points: one `{id}_merged.shp`.
- **Extent polygon grid** — vector polygons: the 300-arcsec grid.

**Graph**

1. **Create spatial index** on the PS points (`Vector / Data Management Tools / Create Spatial Index`).
2. **Extract layer extent** on that indexed layer (`Vector / Research Tools / Extract Layer Extent`). Output is one polygon covering the points.
3. **Buffer** that extent (`Vector / Geoprocessing Tools / Buffer`). Distance **`0.005`** in **layer CRS units**. EGMS L2b shapefiles are geographic (typically EPSG:4326), so this is **0.005 degrees** (~550 m), not metres. Other buffer settings: segments `5`, end cap **Round**, join **Miter**, miter limit `2`, dissolve result **Yes**.
4. **Create spatial index** on the extent polygon grid.
5. **Extract by location** (`Vector / Research Tools / Extract by Location`): extract from the indexed extent polygon grid; overlay is the buffered PS extent; predicate **intersect**. This keeps only the 300-arcsec cells that overlap the PS area.
6. **Create grid** (`Vector / Research Tools / Create Grid`) on the extent of those extracted cells. Type **rectangle (polygon)**; horizontal and vertical spacing **`0.000833333333`** (3 arcsec); CRS is the **project CRS** (same geographic CRS as the points).
7. **Count points in polygon** (`Vector / Analysis Tools / Count Points in Polygon`): polygons are the fine grid from step 6; points are the indexed PS layer from step 1; count field name **`NUMPOINTS`**.
8. Model output: that polygon layer, named **`output_point_density_vector`**.

**After the model**

Rasterize the `NUMPOINTS` field **outside** the modeler (**Raster / Conversion / Rasterize (Vector to Raster)**). Pixel size must be `0.000833333333`. The modeler rounds to `0.000832999` and the pixels no longer line up with the coherence grid.

### Combine ascending and descending

1. **Raster / Merge**. Overlapping pixels keep the value from the last raster in the list. The paper used the western raster for those few overlapping pixels.
2. **GDAL / Raster conversion / Translate (convert format)**: set nodata to `-1` so pixels with zero PS points stay in later statistics.

The training mask folder is `mask_egms_dir` as defined in config/paths.yaml.